In [1]:
import os
import pandas as pd
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from gensim.models import FastText

In [2]:
os.chdir("../")

In [3]:
from src.functions import remove_stopwords_punctuation, remove_outliers, lematiza_tokens, embedding_ft, salva_embeddings

In [4]:
df = pd.read_csv("./data/buscape.csv")

In [5]:
df

,original_index,review_text,review_text_processed,review_text_tokenized,polarity,rating,kfold_polarity,kfold_rating
0,4_55516,"Estou muito satisfeito, o visor é melhor do qu...","estou muito satisfeito, o visor e melhor do qu...","['estou', 'muito', 'satisfeito', 'visor', 'mel...",1.0,4,1,1
1,minus_1_105339,"""muito boa\n\nO que gostei: preco\n\nO que não...","""muito boa\n\no que gostei: preco\n\no que nao...","['muito', 'boa', 'que', 'gostei', 'preco', 'qu...",1.0,5,1,1
2,23_382139,"Rápida, ótima qualidade de impressão e fácil d...","rapida, otima qualidade de impressao e facil d...","['rapida', 'otima', 'qualidade', 'de', 'impres...",1.0,5,1,1
3,2_446456,Produto de ótima qualidade em todos os quesito!,produto de otima qualidade em todos os quesito!,"['produto', 'de', 'otima', 'qualidade', 'em', ...",1.0,5,1,1
4,0_11324,Precisava comprar uma tv compatível com meu dv...,precisava comprar uma tv compativel com meu dv...,"['precisava', 'comprar', 'uma', 'tv', 'compati...",1.0,5,1,1
...,...,...,...,...,...,...,...,...
84986,1_422965,"Produto muito bom, simples e barato","produto muito bom, simples e barato","['produto', 'muito', 'bom', 'simples', 'barato']",1.0,5,10,10
84987,minus_1_150466,O esquema antigo de desmontagem e limpeza das ...,o esquema antigo de desmontagem e limpeza das ...,"['esquema', 'antigo', 'de', 'desmontagem', 'li...",NaN,3,-1,10
84988,0_414799,Esse jogo é muito maneiro é um jogo onde vc te...,esse jogo e muito maneiro e um jogo onde vc te...,"['esse', 'jogo', 'muito', 'maneiro', 'um', 'jo...",1.0,5,10,10
84989,0_389898,Muito bom e intuitivo!\n\nO que gostei: Educa ...,muito bom e intuitivo!\n\no que gostei: educa ...,"['muito', 'bom', 'intuitivo', 'que', 'gostei',...",NaN,3,-1,10


### 1. Train-Test Split

In [6]:
X_data = df.drop(columns=['polarity'])
y_data = df['polarity']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

## Conjunto de Treino

### 2. Limpeza de Dados
##### 2.1. Tratamento de Nulos

In [8]:
X_train.shape

(67992, 7)

In [9]:
X_train['review_text'].isnull().sum()

np.int64(1)

In [10]:
X_train = X_train.dropna(subset=['review_text'])

In [11]:
X_train.shape

(67991, 7)

In [12]:
# paridade de índices
y_train = y_train.loc[X_train.index]

In [13]:
y_train.shape

(67991,)

##### 2.2. Tratamento de Outliers

In [14]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [15]:
X_train_token = pd.DataFrame()
X_train_token['token'] = X_train['review_text']
X_train_token

,token
54310,"Em relação ao preço o produto está adequado, m..."
68526,Bom!
43681,eu já estava satisfeito com as suas funcionali...
49718,"Fácil de se instalar, de se manusear, barato, ..."
62727,jogo muito bom melhor jogo que eu ja vi depois...
...,...
6265,acho lindo quero esse modelo de qualquer geito...
54886,todomundo vai querer um ipad como esse
76820,"foi um investimento muito bom, sem arrependime..."
860,"Considero um bom aparelho, com um conceito eco..."


In [16]:
X_train_token['token'] = X_train_token['token'].apply(word_tokenize)

In [17]:
X_train_token['token'] = X_train_token['token'].apply(lambda text: remove_stopwords_punctuation(text))

In [18]:
X_train_token['len_token'] = X_train_token['token'].apply(lambda text: len(text))

In [19]:
X_train_token

,token,len_token
54310,"[Em, relação, preço, produto, adequado, concor...",38
68526,[Bom],1
43681,"[satisfeito, funcionalidades, tentei, comparar...",15
49718,"[Fácil, instalar, manusear, barato, boa, quali...",22
62727,"[jogo, bom, melhor, jogo, ja, vi, gta, iv, gta...",12
...,...,...
6265,"[acho, lindo, quero, modelo, qualquer, geito, ...",19
54886,"[todomundo, vai, querer, ipad]",4
76820,"[investimento, bom, arrependimento, O, gostei,...",25
860,"[Considero, bom, aparelho, conceito, ecológico...",25


In [20]:
X_train_token = remove_outliers(X_train_token, 'len_token')

In [21]:
X_train_token

,token,len_token
54310,"[Em, relação, preço, produto, adequado, concor...",38
68526,[Bom],1
43681,"[satisfeito, funcionalidades, tentei, comparar...",15
49718,"[Fácil, instalar, manusear, barato, boa, quali...",22
62727,"[jogo, bom, melhor, jogo, ja, vi, gta, iv, gta...",12
...,...,...
6265,"[acho, lindo, quero, modelo, qualquer, geito, ...",19
54886,"[todomundo, vai, querer, ipad]",4
76820,"[investimento, bom, arrependimento, O, gostei,...",25
860,"[Considero, bom, aparelho, conceito, ecológico...",25


In [22]:
# paridade de índices em X e y
X_train = X_train.loc[X_train_token.index]

In [23]:
y_train = y_train.loc[X_train_token.index]

In [24]:
print(X_train.shape, y_train.shape)

(62998, 7) (62998,)


### 3. Transformação
##### 3.1. Remoção de colunas "inúteis"

In [25]:
X_train.columns

Index(['original_index', 'review_text', 'review_text_processed',
       'review_text_tokenized', 'rating', 'kfold_polarity', 'kfold_rating'],
      dtype='str')

In [26]:
X_train = X_train.drop(columns=['original_index', 'review_text_processed', 'review_text_tokenized', 'rating', 'kfold_polarity', 'kfold_rating'])

In [27]:
print(f'colunas: {X_train.columns}. tipo: {type(X_train)}')

colunas: Index(['review_text'], dtype='str'). tipo: <class 'pandas.DataFrame'>


##### 3.2. Tratamento na label

- Transforma coluna binária em ternária

In [28]:
y_train.value_counts()

polarity
1.0    50283
0.0     4486
Name: count, dtype: int64

In [29]:
y_train.isnull().sum()

np.int64(8229)

In [30]:
y_train = y_train.map({1.0: 2, 0.0: 0})  # {valor_antigo: valor_atual}
y_train = y_train.fillna(1)

In [31]:
y_train.value_counts()

polarity
2.0    50283
1.0     8229
0.0     4486
Name: count, dtype: int64

##### 3.3. Resampling

In [32]:
type(y_train)

pandas.Series

In [33]:
us = RandomUnderSampler(random_state=0)

In [34]:
X_train, y_train = us.fit_resample(X_train, y_train)

In [35]:
print(X_train.shape, y_train.shape, type(X_train), type(y_train))

(13458, 1) (13458,) <class 'pandas.DataFrame'> <class 'pandas.Series'>


In [36]:
y_train.value_counts()

polarity
0.0    4486
1.0    4486
2.0    4486
Name: count, dtype: int64

##### 3.4. PROCESSAMENTO DE LINGUAGEM NATURAL
##### 3.4.1. Tokenização

In [37]:
X_train

,review_text
32045,"Comprei um que veio sem o controle remoto, foi..."
65876,Grande engodo. Não recomendo nem para os inimi...
45709,"MUITO RUIM, POIS NÃO USO MAIS O PRODUTO, ESTÁ ..."
11067,não valeu o investimento\n\nO que gostei: qual...
10193,Quem é doido de comprar um HT com esse preço? ...
...,...
48505,"No geral estou muito satisfeito, embora com po..."
76302,"muito bom ,o ruim é o preço"
19394,os prudutos barbie deveriam ser mais baratos\n...
4544,O jogo passa-se na cidade de Washington D.C. e...


In [38]:
X_train['review_text'] = X_train['review_text'].apply(word_tokenize)

In [39]:
X_train

,review_text
32045,"[Comprei, um, que, veio, sem, o, controle, rem..."
65876,"[Grande, engodo, ., Não, recomendo, nem, para,..."
45709,"[MUITO, RUIM, ,, POIS, NÃO, USO, MAIS, O, PROD..."
11067,"[não, valeu, o, investimento, O, que, gostei, ..."
10193,"[Quem, é, doido, de, comprar, um, HT, com, ess..."
...,...
48505,"[No, geral, estou, muito, satisfeito, ,, embor..."
76302,"[muito, bom, ,, o, ruim, é, o, preço]"
19394,"[os, prudutos, barbie, deveriam, ser, mais, ba..."
4544,"[O, jogo, passa-se, na, cidade, de, Washington..."


##### 3.4.2. Remoção de stopwords e pontuação

In [40]:
X_train['review_text'] = X_train['review_text'].apply(lambda x: remove_stopwords_punctuation(x))

In [41]:
X_train

,review_text
32045,"[Comprei, veio, controle, remoto, briga, conse..."
65876,"[Grande, engodo, Não, recomendo, inimigos, O, ..."
45709,"[MUITO, RUIM, POIS, NÃO, USO, MAIS, O, PRODUTO..."
11067,"[valeu, investimento, O, gostei, qualidade, so..."
10193,"[Quem, doido, comprar, HT, preço, Tudo, bem, m..."
...,...
48505,"[No, geral, satisfeito, embora, poucos, dias, ..."
76302,"[bom, ruim, preço]"
19394,"[prudutos, barbie, deveriam, baratos, O, goste..."
4544,"[O, jogo, passa-se, cidade, Washington, D.C., ..."


##### 3.4.3. Lemmatizing

In [42]:
!python -m spacy download pt_core_news_sm

     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
      --------------------------------------- 0.3/13.0 MB ? eta -:--:--
     --------- ------------------------------ 3.1/13.0 MB 13.2 MB/s eta 0:00:01
     ---------------------- ----------------- 7.3/13.0 MB 16.8 MB/s eta 0:00:01
     --------------------------------------  12.8/13.0 MB 19.7 MB/s eta 0:00:01
     --------------------------------------- 13.0/13.0 MB 17.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
nlp = spacy.load("pt_core_news_sm")

In [44]:
X_train["review_text"] = X_train["review_text"].apply(lambda x: lematiza_tokens(x, nlp))

In [45]:
X_train

,review_text
32045,"[Comprei, vir, controle, remoto, briga, conseg..."
65876,"[grande, engodo, não, recomendar, inimigo, o, ..."
45709,"[MUITO, RUIM, POIS, NÃO, USO, MAIS, o, PRODUTO..."
11067,"[valer, investimento, o, gostar, qualidade, so..."
10193,"[quem, doir, comprar, HT, preço, tudo, bem, ma..."
...,...
48505,"[em o, geral, satisfeito, embora, pouco, dia, ..."
76302,"[bom, ruim, preço]"
19394,"[pruduto, barbie, dever, barato, o, gostar, ne..."
4544,"[o, jogo, passar se, cidade, Washington, D.C.,..."


##### 3.4.4. Embedding

In [46]:
# FastText: representa a palavra pela soma dos n-gramas de caracteres, então consegue vetorizar palavras inéditas do conjunto de teste
model_embedding = FastText(X_train['review_text'], min_count=1, vector_size=100, window=5)

In [47]:
X_train['review_text'] = X_train['review_text'].apply(lambda x: embedding_ft(x, model_embedding))

In [48]:
X_train = X_train.rename(columns={'review_text': 'review_embedding'})

In [49]:
X_train

,review_embedding
32045,"[[0.541763, 1.5301492, -0.008526714, 0.9061262..."
65876,"[[0.15286225, 1.042737, 0.44231987, 0.80964196..."
45709,"[[0.2772419, 2.4906337, -0.13765459, 1.8087661..."
11067,"[[-0.7942487, 2.1369376, -0.64485306, 2.224255..."
10193,"[[0.4941866, 1.6064278, 0.31946585, 0.8899307,..."
...,...
48505,"[[0.23655686, 1.2284592, 0.21861248, 0.5818474..."
76302,"[[-1.3517793, 0.2535928, 0.44723675, 1.9848949..."
19394,"[[-0.027987447, 0.59719825, 0.22105311, 0.5766..."
4544,"[[-2.7770128, -2.0675735, -0.61283684, 2.83831..."


### 4. Salvando os dados

In [50]:
salva_embeddings(X_train, y_train, 'data/X_train.pt', 'data/y_train.pt', vector_size=100)

c:\Users\roger\OneDrive\Documentos\Coding Repos\NLP-Sentiment-Classification-PyTorch\src\functions.py:65: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:255.)
  torch.tensor(row, dtype=torch.float32)


## Conjunto de Teste

O conjunto de teste recebe **só as transformações que o modelo precisa para conseguir ler o dado** — nada que altere a distribuição, porque ele tem que representar o mundo real na hora da avaliação.

| Etapa do treino | Aplicada no teste? | Motivo |
|---|---|---|
| Tratamento de nulos | SIM | `word_tokenize` quebra com `NaN` |
| Remoção de outliers (`len_token`) | NÃO | descartar reviews longas maquiaria a métrica |
| Remoção de colunas | SIM | o modelo espera as mesmas features |
| Mapeamento da label | SIM | os rótulos têm que significar o mesmo nos dois conjuntos |
| Undersampling | NÃO | a avaliação tem que ser na distribuição real (desbalanceada) |
| Tokenização / stopwords / lemmatizing | SIM | mesmo pré-processamento de texto |
| Embedding | SIM *(modelo de treino, sem re-treinar)* | re-treinar seria vazamento e mudaria o espaço vetorial |

### 2. Limpeza de Dados
##### 2.1. Tratamento de Nulos

In [51]:
print(X_test.shape)
X_test['review_text'].isnull().sum()

(16999, 7)


np.int64(0)

In [52]:
# remove nulos
X_test = X_test.dropna(subset=['review_text'])

# paridade de índices
y_test = y_test.loc[X_test.index]

In [53]:
print(X_test.shape, y_test.shape)

(16999, 7) (16999,)


### 3. Transformação
##### 3.1. Remoção de colunas "inúteis"

In [54]:
X_test = X_test.drop(columns=['original_index', 'review_text_processed', 'review_text_tokenized', 'rating', 'kfold_polarity', 'kfold_rating'])
print(f'colunas: {X_test.columns}. tipo: {type(X_test)}')

colunas: Index(['review_text'], dtype='str'). tipo: <class 'pandas.DataFrame'>


##### 3.2. Tratamento na label

In [55]:
y_test = y_test.map({1.0: 2, 0.0: 0})  # {valor_antigo: valor_atual}
y_test = y_test.fillna(1)

In [56]:
y_test.value_counts()

polarity
2.0    13485
1.0     2190
0.0     1324
Name: count, dtype: int64

##### 3.3. PROCESSAMENTO DE LINGUAGEM NATURAL
##### 3.3.1. Tokenização

In [57]:
X_test['review_text'] = X_test['review_text'].apply(word_tokenize)

##### 3.3.2. Remoção de stopwords e pontuação

In [58]:
X_test['review_text'] = X_test['review_text'].apply(lambda x: remove_stopwords_punctuation(x))

##### 3.3.3. Lemmatizing

Reaproveita o `nlp` (`pt_core_news_sm`) já carregado na seção de treino.

In [59]:
X_test["review_text"] = X_test["review_text"].apply(lambda x: lematiza_tokens(x, nlp))

##### 3.3.4. Embedding

Usa `model_embedding` **treinado no conjunto de treino** — treinar um novo modelo seria vazamento de dados e resultaria em um espaço vetorial incompatível com o do treino.

In [60]:
X_test['review_text'] = X_test['review_text'].apply(lambda x: embedding_ft(x, model_embedding))
X_test = X_test.rename(columns={'review_text': 'review_embedding'})

In [61]:
X_test

,review_embedding
36053,"[[-0.14380212, 0.55430424, 0.46279097, 1.00813..."
55549,"[[-0.8460861, 0.36209136, 0.5605696, 1.1338792..."
51600,"[[1.5155655, 2.400585, 1.58224, 1.5824039, 0.1..."
12053,"[[0.1657444, 1.0386283, 0.16049361, 0.7603164,..."
59505,"[[0.1298283, 0.5476991, 0.16379064, 0.38655025..."
...,...
63113,"[[-0.5148737, 0.5504312, 0.47709116, 0.9876389..."
64731,"[[-0.13453323, 0.5510868, 0.61678064, 0.614764..."
35535,"[[-2.7770128, -2.0675735, -0.61283684, 2.83831..."
58210,"[[-0.3584738, 0.64138377, 0.40784404, 1.089948..."


### 4. Salvando os dados

In [62]:
salva_embeddings(X_test, y_test, 'data/X_test.pt', 'data/y_test.pt', vector_size=100)